In [2]:
from PIL import Image

def embed_data(image_path, binary_secret, output_path):
    img = Image.open(image_path)
    img = img.convert('RGB')
    pixels = img.load()      
    width, height = img.size
    
    # คำนวณความยาวของข้อมูล แล้วแปลงความยาวนั้นเป็นเลขฐานสอง (ขนาด 32 บิต)
    data_len = len(binary_secret)
    length_header = format(data_len, '032b') 
    
    # เอา "ความยาว" ไปต่อหน้า "ข้อมูลจริง"
    full_binary = length_header + binary_secret
    
    max_capacity = width * height * 3 
    if len(full_binary) > max_capacity:
        print(f"❌ ภาพหลักเล็กเกินไป! ต้องการ {len(full_binary)} บิต แต่จุได้แค่ {max_capacity} บิต")
        return False

    bit_index = 0
    total_len = len(full_binary)

    for y in range(height):
        for x in range(width):
            if bit_index < total_len:
                r, g, b = pixels[x, y]
                
                if bit_index < total_len:
                    r = r - (r % 2) + int(full_binary[bit_index])
                    bit_index += 1
                if bit_index < total_len:
                    g = g - (g % 2) + int(full_binary[bit_index])
                    bit_index += 1
                if bit_index < total_len:
                    b = b - (b % 2) + int(full_binary[bit_index])
                    bit_index += 1

                pixels[x, y] = (r, g, b)
            else:
                break
                
    img.save(output_path)
    print("✅ ฝังข้อมูลสำเร็จ! ได้ภาพใหม่ชื่อ:", output_path)
    return True

if __name__ == "__main__":
    print("=== โปรเจกต์ Shadow-Pixel: ระบบฝังข้อมูล ===")
    print("เลือกสิ่งที่คุณต้องการซ่อน:")
    print("1. ซ่อนข้อความลับ (Text)")
    print("2. ซ่อนรูปภาพลับ (Image - catcat.png)")
    choice = input("พิมพ์ 1 หรือ 2 : ")
    
    # ที่อยู่ไฟล์ในเครื่องของคุณ
    original_image = r"D:\Users\ASUS\Desktop\mini-root\cat.png"        
    stego_image = r"D:\Users\ASUS\Desktop\mini-root\output_cat.png"    
    secret_img_path = r"D:\Users\ASUS\Desktop\mini-root\catcat.png"

    if choice == "1":
        secret_message = input("พิมพ์ข้อความที่ต้องการซ่อน: ")
        data_bytes = ("TXT|" + secret_message).encode('utf-8')
        
    elif choice == "2":
        try:
            with open(secret_img_path, "rb") as f:
                img_data = f.read()
            data_bytes = b"IMG|" + img_data
            print(f"📦 อ่านไฟล์รูปภาพลับสำเร็จ! เตรียมตัวซ่อน...")
        except FileNotFoundError:
            print(f"❌ หาไฟล์รูปภาพลับไม่เจอครับ")
            exit()
    else:
        print("เลือกผิดหมวดหมู่ครับ จบการทำงาน")
        exit()

    # แปลงข้อมูลเป็น 0101
    binary_data = ''.join(format(byte, '08b') for byte in data_bytes)
    
    print("\n--- กำลังเริ่มกระบวนการฝังรหัสลงในภาพ ---")
    embed_data(original_image, binary_data, stego_image)

=== โปรเจกต์ Shadow-Pixel: ระบบฝังข้อมูล ===
เลือกสิ่งที่คุณต้องการซ่อน:
1. ซ่อนข้อความลับ (Text)
2. ซ่อนรูปภาพลับ (Image - catcat.png)
📦 อ่านไฟล์รูปภาพลับสำเร็จ! เตรียมตัวซ่อน...

--- กำลังเริ่มกระบวนการฝังรหัสลงในภาพ ---
✅ ฝังข้อมูลสำเร็จ! ได้ภาพใหม่ชื่อ: D:\Users\ASUS\Desktop\mini-root\output_cat.png
